# UEBA Synthetic Data Generation & Analysis
This notebook generates a multi-dimensional dataset for a User and Entity Behavior Analytics (UEBA) system. It simulates normal user behavior and injects realistic anomalies (hacks, bots, and data exfiltration) at a 3% attack rate.

## Data Dictionary
The data is split across three modules, simulating a microservice logging architecture:

| Log File | Features Generated | Purpose |
| :--- | :--- | :--- |
| **`log_login`** | `login_id`, `user_id`, `timestamp`, `country`, `city`, `user_agent`, `os`, `resolution`, `login_status`, `auth_method`, `is_anomaly` | Tracks authentication context. Acts as the master table containing the ground-truth `is_anomaly` label. |
| **`log_behavior`** | `login_id`, `user_id`, `mouse_velocity_px_sec`, `avg_keystroke_delay`, `session_entropy`, `tab_switch_count` | Captures in-session human-computer interaction (HCI) metrics to detect bot/script activity. |
| **`log_network`** | `login_id`, `user_id`, `bytes_sent`, `destination_port`, `protocol`, `packet_loss_rate` | Monitors backend traffic to detect scanning or data exfiltration. |

Imports & Data Generation

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
Faker.seed(42)
random.seed(42)

# --- CONFIGURATION ---
NUM_USERS = 500
DAYS_OF_DATA = 30
ANOMALY_RATE = 0.03 # 3% Attack Rate (Realistic is low)

# --- 1. DIVERSE DEVICE POOL (Real UA Strings) ---
REAL_USER_AGENTS = {
    "Windows": [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:89.0) Gecko/20100101 Firefox/89.0",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Edge/91.0.864.59",
        "Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36" 
    ],
    "MacOS": [
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.114 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Safari/605.1.15"
    ],
    "Linux": [
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.101 Safari/537.36",
        "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:89.0) Gecko/20100101 Firefox/89.0"
    ],
    "iOS": [
        "Mozilla/5.0 (iPhone; CPU iPhone OS 14_6 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0 Mobile/15E148 Safari/604.1",
        "Mozilla/5.0 (iPad; CPU OS 14_6 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0 Mobile/15E148 Safari/604.1"
    ],
    "Android": [
        "Mozilla/5.0 (Linux; Android 11; SM-G991B) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.120 Mobile Safari/537.36",
        "Mozilla/5.0 (Linux; Android 10; K) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.120 Mobile Safari/537.36"
    ]
}

RESOLUTIONS = ["1920x1080", "1366x768", "1440x900", "1536x864", "2560x1440", "390x844", "414x896"]

# --- 2. GENERATE USER PROFILES ---
user_db = {}
for i in range(NUM_USERS):
    user_id = f"user_{i+1:03d}"
    home_country = random.choice(['US', 'GB', 'DE', 'FR', 'IN', 'CA', 'AU'])
    local = fake.local_latlng(country_code=home_country)
    
    primary_os = random.choice(list(REAL_USER_AGENTS.keys()))
    primary_ua = random.choice(REAL_USER_AGENTS[primary_os])
    
    user_db[user_id] = {
        "home_country": home_country,
        "home_city": local[2],
        "home_lat": float(local[0]),
        "home_lon": float(local[1]),
        "usual_os": primary_os,
        "usual_ua": primary_ua,
        "usual_res": random.choice(RESOLUTIONS),
        "avg_txn": random.randint(20, 1000),
        "avg_typing_speed": random.uniform(0.15, 0.35) 
    }

# --- 3. GENERATE LOGS ---
data_login, data_behavior, data_network = [], [], []
print(f"Generating data for {NUM_USERS} users...")

start_date = datetime.now() - timedelta(days=DAYS_OF_DATA)

for user_id in user_db:
    profile = user_db[user_id]
    current_time = start_date
    
    while current_time < datetime.now():
        login_id = fake.uuid4()
        is_attack = random.random() < ANOMALY_RATE
        
        # --- MODULE 1: LOGIN ---
        if is_attack:
            if random.random() > 0.5:
                ua_string = "Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)" 
                os_type = "Bot"
                res = "800x600"
            else:
                os_type = "Windows"
                ua_string = random.choice(REAL_USER_AGENTS["Windows"])
                res = "1024x768"

            country = random.choice(['RU', 'CN', 'KP', 'BR', 'NG'])
            city = fake.city()
            login_status = random.choice(["Failed", "Failed", "Success"]) 
            auth_method = "Password"
        else:
            if random.random() > 0.9:
                os_type = "iOS" if profile["usual_os"] != "iOS" else "Windows"
                ua_string = random.choice(REAL_USER_AGENTS[os_type])
                res = "390x844" if os_type == "iOS" else "1920x1080"
            else:
                os_type = profile["usual_os"]
                ua_string = profile["usual_ua"]
                res = profile["usual_res"]

            country = profile["home_country"]
            city = profile["home_city"]
            login_status = "Success"
            auth_method = "Biometric" if os_type in ["iOS", "Android"] else "Password"

        data_login.append({
            "login_id": login_id, "user_id": user_id, "timestamp": current_time,
            "country": country, "city": city, "user_agent": ua_string,      
            "os": os_type, "resolution": res, "login_status": login_status,
            "auth_method": auth_method, "is_anomaly": 1 if is_attack else 0
        })

        # --- MODULE 2: BEHAVIOR ---
        if is_attack:
            mouse_speed = random.randint(4000, 8000) 
            keystroke = 0.005 
            entropy = 0.1 
        else:
            mouse_speed = int(np.random.normal(300, 100)) 
            keystroke = np.random.normal(profile["avg_typing_speed"], 0.05)
            entropy = random.uniform(0.6, 0.95) 

        data_behavior.append({
            "login_id": login_id, "user_id": user_id,
            "mouse_velocity_px_sec": abs(mouse_speed), "avg_keystroke_delay": abs(round(keystroke, 4)),
            "session_entropy": round(entropy, 2), "tab_switch_count": random.randint(0, 10)
        })

        # --- MODULE 3: NETWORK ---
        if is_attack:
            bytes_out = random.randint(10_000_000, 100_000_000) 
            port = random.choice([21, 22, 3389]) 
            protocol = "UDP" 
        else:
            bytes_out = random.randint(1000, 150000) 
            port = 443 
            protocol = "TCP"

        data_network.append({
            "login_id": login_id, "user_id": user_id, "bytes_sent": bytes_out,
            "destination_port": port, "protocol": protocol,
            "packet_loss_rate": round(random.uniform(0, 0.05), 3)
        })

        current_time += timedelta(hours=random.randint(4, 72))

# --- SAVE FILES ---
df_login = pd.DataFrame(data_login)
df_behav = pd.DataFrame(data_behavior)
df_net = pd.DataFrame(data_network)

df_login.to_csv("log_login_500users.csv", index=False)
df_behav.to_csv("log_behavior_500users.csv", index=False)
df_net.to_csv("log_network_500users.csv", index=False)

print(f"DONE. Generated 3 CSVs.")
print(f"Total Logins: {len(df_login)}")
print(f"Attacks Simulated: {df_login['is_anomaly'].sum()}")

In [ ]:
Visualizing the Anomalies

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="darkgrid")
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('UEBA Threat Analysis Signatures', fontsize=18, fontweight='bold')

# We need the 'is_anomaly' label from the login table to color our behavior and network charts
df_behav_merged = pd.merge(df_behav, df_login[['login_id', 'is_anomaly']], on='login_id')
df_net_merged = pd.merge(df_net, df_login[['login_id', 'is_anomaly']], on='login_id')

# Map the binary labels to readable strings for the legend
label_map = {0: 'Normal Human', 1: 'Malicious Bot/Hacker'}

# --- GRAPH 1: Login OS Distribution ---
# Shows how bots spoof OS or use headless browsers
sns.countplot(ax=axes[0], data=df_login, x='os', hue=df_login['is_anomaly'].map(label_map), palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Login Requests by Operating System')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Detected OS')
axes[0].tick_params(axis='x', rotation=45)

# --- GRAPH 2: Behavior Module Clustering ---
# A scatter plot is best to show how bots group together 
sns.scatterplot(ax=axes[1], data=df_behav_merged, 
                x='avg_keystroke_delay', y='mouse_velocity_px_sec', 
                hue=df_behav_merged['is_anomaly'].map(label_map), 
                palette=['#2ecc71', '#e74c3c'], alpha=0.7)
axes[1].set_title('Human vs. Bot HCI Patterns')
axes[1].set_xlabel('Avg Keystroke Delay (Seconds)')
axes[1].set_ylabel('Mouse Velocity (Pixels/Sec)')

# --- GRAPH 3: Network Data Exfiltration ---
# A box plot perfectly visualizes the massive spike in bytes sent during an attack 
sns.boxplot(ax=axes[2], data=df_net_merged, 
            x=df_net_merged['is_anomaly'].map(label_map), y='bytes_sent', 
            palette=['#2ecc71', '#e74c3c'])
axes[2].set_title('Network Traffic Volume (Data Exfiltration)')
axes[2].set_ylabel('Bytes Sent')
axes[2].set_xlabel('Session Type')
axes[2].set_yscale("log") # Using log scale because the attack spikes are massive

plt.tight_layout()
plt.show()